# Table 3 — CIE Exact-Match (QID-level P/R/F1)

Computes QID-level set-comparison P/R/F1 for models that produce Wikidata QIDs
(e.g. `relik-cie`). This is a **separate table** from the similarity-based metrics.

## Setup

Run from a salloc on rome (no GPU needed, ~16GB RAM):
```
salloc -p rome -n 1 --ntasks-per-node 1 --cpus-per-task 1 -t 1:59:00 --mem=16G
conda activate emerge
cd ~/repositories/emerge-stage
jupyter lab --no-browser --port=8888
```

In [1]:
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

from stats.evaluation.load_results import (
    load_results,
    models_to_load,
    model_name_to_latex,
    tkgu_type_to_latex,
    arch_color_for_model,
)

In [2]:
# --- CONFIGURE THIS ---
# Scoring modes:
# - Legacy (reproduces ICML submission): use *_legacy_with_kg pkl
# - Fixed (post-rebuttal, score_empty_predictions_as_zero=true): use *_fixed_with_kg pkl
# See src/evaluation/README.md for details on the scoring bug.

# --- Fixed scoring + KG snapshots:
PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_submitted_icml_fixed_with_kg/wiki_eval_result.pkl'
# --- Legacy scoring + KG snapshots (reproduces paper values):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_submitted_icml_legacy_with_kg/wiki_eval_result.pkl'
# --- Fixed scoring, no KG (relik-cie x-triples missing):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260217_submitted_icml_fixed/wiki_eval_result.pkl'

assessor_by_prompt_type = {
    'triple_assertion': 'Meta-Llama-3.1-405B_prompt_v1',
    'triple_deprecation': 'Meta-Llama-3.1-405B_prompt_v1'
}

In [3]:
(df_wiki_metrics_cie, df_metrics_open_ie, df_preds_gt_cie,
 df_preds_open_ie, df_instances, df_additional_stats) = load_results(
    pkl_path=PKL_PATH,
    assessor_by_prompt_type=assessor_by_prompt_type,
    filter_models=models_to_load,
)

print(f'df_preds_gt_cie.shape: {df_preds_gt_cie.shape}')
print(f'triple_types: {df_preds_gt_cie.triple_type.unique()}')

df_wiki_metrics_cie.shape: (5066401, 10)
df_wiki_metrics_cie_filtered.shape: (4923159, 10)
df_preds_gt_cie.shape: (56686, 24)
df_preds_gt_cie_filtered.shape: (43563, 25)


df_preds_gt_cie.shape: (43563, 27)
triple_types: ['in-dataset' 'predicted']


## Compute QID exact-match P/R/F1

For each model and tkgu_type, compare the set of predicted triples (QID tuples)
against the set of GT triples. Compute set-based precision, recall, F1.

In [4]:
# Which models to include (only CIE models that produce QIDs)
allowed_models = [
    'relik-cie',
    # Add other QID-producing models here if needed
]

SHOW_EMPTY_AS_DASH = True

tkgu_type_rank = {
    "x-triples": 0,
    "e-triples": 1,
    "ee-triples": 2,
    "ee-kg-triples": 3,
    "d-triples": 4,
}

df = df_preds_gt_cie.copy()

# Split GT vs Predictions
gt = df[df["triple_type"] == "in-dataset"].copy()
pred = df[df["triple_type"] == "predicted"].copy()

key_cols = ["hash_id", "tkgu_type", "triple_head", "triple_relation", "triple_tail"]
gt = gt.dropna(subset=key_cols)
pred = pred.dropna(subset=key_cols + ["model"])
pred = pred[pred["model"].isin(allowed_models)].copy()

def make_key(d):
    return (
        d["hash_id"].astype(str) + "||" +
        d["tkgu_type"].astype(str) + "||" +
        d["triple_head"].astype(str) + "||" +
        d["triple_relation"].astype(str) + "||" +
        d["triple_tail"].astype(str)
    )

gt["_key"] = make_key(gt)
pred["_key"] = make_key(pred)

# tkgu types from predictions
tkgu_types_present = pred["tkgu_type"].dropna().unique().tolist()
tkgu_order = sorted(
    tkgu_types_present,
    key=lambda t: (tkgu_type_rank.get(t, 10**9), tkgu_types_present.index(t))
)

gt_by_type = gt.groupby("tkgu_type")["_key"].apply(lambda s: set(s.unique())).to_dict()
pred_by_model_type = (
    pred.groupby(["model", "tkgu_type"])["_key"]
    .apply(lambda s: set(s.unique()))
    .to_dict()
)

models_present = sorted(pred["model"].dropna().unique().tolist())

records = []
for model in models_present:
    for t in tkgu_order:
        gt_set = gt_by_type.get(t, set())
        pr_set = pred_by_model_type.get((model, t), set())

        tp = len(gt_set & pr_set)
        fp = len(pr_set - gt_set)
        fn = len(gt_set - pr_set)

        if SHOW_EMPTY_AS_DASH and (len(pr_set) == 0 or len(gt_set) == 0):
            prec = rec = f1 = float("nan")
        else:
            prec = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
            rec  = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
            f1   = (2 * prec * rec / (prec + rec)) if (pd.notna(prec) and pd.notna(rec) and (prec + rec) > 0) else float("nan")

        records.append({
            "model": model,
            "tkgu_type": t,
            "P": prec * 100 if pd.notna(prec) else float("nan"),
            "R": rec * 100 if pd.notna(rec) else float("nan"),
            "F1": f1 * 100 if pd.notna(f1) else float("nan"),
        })

metrics = pd.DataFrame(records)
print(f'Computed metrics for {len(models_present)} models, {len(tkgu_order)} tkgu_types')
metrics

Computed metrics for 1 models, 2 tkgu_types


,model,tkgu_type,P,R,F1
0,relik-cie,x-triples,34.068763,14.495815,20.338053
1,relik-cie,e-triples,2.046394,13.482653,3.553446


## Generate LaTeX table

In [5]:
metric_order = ["P", "R", "F1"]

table_num = metrics.pivot_table(
    index="model",
    columns="tkgu_type",
    values=["P", "R", "F1"],
    aggfunc="mean",
)

table_num = table_num.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)
desired_cols = [(t, m) for t in tkgu_order for m in metric_order]
table_num = table_num.reindex(
    columns=pd.MultiIndex.from_tuples(desired_cols, names=["TKGU", "metric"]))

# Pretty model names
table_num.index = table_num.index.map(lambda m: model_name_to_latex.get(m, m))

def fmt_cell(x):
    if pd.isna(x):
        return "--"
    return f"{x:.1f}"

table = table_num.apply(lambda col: col.map(fmt_cell)).sort_index()

# LaTeX body with colored model cells
body = table.to_latex(index=True, header=False, escape=False)
body_lines = body.splitlines()
body_content = "\n".join(body_lines[2:-2])

colored_lines = []
for line in body_content.splitlines():
    if " & " not in line:
        colored_lines.append(line)
        continue
    first, rest = line.split(" & ", 1)
    indent = line[:len(line) - len(line.lstrip(" "))]
    model_cell = first.strip()
    colored_first = f"{indent}\\cellcolor{{{arch_color_for_model(model_cell)}}} {model_cell}"
    colored_lines.append(colored_first + " & " + rest)
body_content = "\n".join(colored_lines)

# Headers
tkgu_order_hdr = [tkgu_type_to_latex.get(t, t) for t in tkgu_order]

header_top_parts = []
for i, t in enumerate(tkgu_order_hdr):
    bar = "|" if i < len(tkgu_order_hdr) - 1 else ""
    header_top_parts.append(f"\\multicolumn{{{len(metric_order)}}}{{c{bar}}}{{{t}}}")
header_top = " & ".join(header_top_parts)

cmid_parts = []
start = 2
for _ in tkgu_order_hdr:
    end = start + len(metric_order) - 1
    cmid_parts.append(f"\\cmidrule(lr){{{start}-{end}}}")
    start = end + 1
cmidrules = "".join(cmid_parts)

header_bottom = " & ".join(metric_order * len(tkgu_order_hdr))

group_spec = "c" * len(metric_order)
tabular_spec = "l|" + "|".join([group_spec] * len(tkgu_order_hdr))

latex = f"""\\begin{{table*}}
\\begin{{center}}
\\begin{{small}}
  \\begin{{sc}}
    \\setlength{{\\tabcolsep}}{{3pt}}
    \\rowcolors{{4}}{{rowwhite}}{{rowgray}}
    \\begin{{tabular}}{{{tabular_spec}}}
      \\toprule
      & {header_top} \\\\
      {cmidrules}
      Model & {header_bottom} \\\\
\\midrule
{body_content}
      \\bottomrule
    \\end{{tabular}}
  \\end{{sc}}
\\end{{small}}
\\end{{center}}
\\vskip -0.1in
\\end{{table*}}"""

print(latex)

\begin{table*}
\begin{center}
\begin{small}
  \begin{sc}
    \setlength{\tabcolsep}{3pt}
    \rowcolors{4}{rowwhite}{rowgray}
    \begin{tabular}{l|ccc|ccc}
      \toprule
      & \multicolumn{3}{c|}{\opexists} & \multicolumn{3}{c}{\opadd} \\
      \cmidrule(lr){2-4}\cmidrule(lr){5-7}
      Model & P & R & F1 & P & R & F1 \\
\midrule
\cellcolor{archOther} model &  &  &  &  &  &  \\
\midrule
\cellcolor{archRELIK} ReLiK cIE & 34.1 & 14.5 & 20.3 & 2.0 & 13.5 & 3.6 \\
      \bottomrule
    \end{tabular}
  \end{sc}
\end{small}
\end{center}
\vskip -0.1in
\end{table*}
